# Build Your Own Mini Framework Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Module Base Class

The abstract interface that every layer implements.

In [ ]:
```python

class Module:

    def __init__(self):

        self.training = True

    def forward(self, x):

        raise NotImplementedError

    def backward(self, grad):

        raise NotImplementedError

    def parameters(self):

        return []

    def train(self):

        self.training = True

    def eval(self):

        self.training = False

In [ ]:
```

### Step 2: Linear Layer

The fundamental building block. Stores weights and biases, computes Wx + b forward, and weight/input gradients backward.

In [ ]:
```python

import math

import random

class Linear(Module):

    def __init__(self, fan_in, fan_out):

        super().__init__()

        std = math.sqrt(2.0 / fan_in)

        self.weights = [[random.gauss(0, std) for _ in range(fan_in)] for _ in range(fan_out)]

        self.biases = [0.0] * fan_out

        self.weight_grads = [[0.0] * fan_in for _ in range(fan_out)]

        self.bias_grads = [0.0] * fan_out

        self.fan_in = fan_in

        self.fan_out = fan_out

        self.input = None

    def forward(self, x):

        self.input = x

        output = []

        for i in range(self.fan_out):

            val = self.biases[i]

            for j in range(self.fan_in):

                val += self.weights[i][j] * x[j]

            output.append(val)

        return output

    def backward(self, grad):

        input_grad = [0.0] * self.fan_in

        for i in range(self.fan_out):

            self.bias_grads[i] += grad[i]

            for j in range(self.fan_in):

                self.weight_grads[i][j] += grad[i] * self.input[j]

                input_grad[j] += grad[i] * self.weights[i][j]

        return input_grad

    def parameters(self):

        params = []

        for i in range(self.fan_out):

            for j in range(self.fan_in):

                params.append((self.weights, i, j, self.weight_grads))

            params.append((self.biases, i, None, self.bias_grads))

        return params

In [ ]:
```

### Step 3: Activation Modules

ReLU, Sigmoid, and Tanh as Modules. Each caches what it needs for the backward pass.

In [ ]:
```python

class ReLU(Module):

    def __init__(self):

        super().__init__()

        self.mask = None

    def forward(self, x):

        self.mask = [1.0 if v > 0 else 0.0 for v in x]

        return [max(0.0, v) for v in x]

    def backward(self, grad):

        return [g * m for g, m in zip(grad, self.mask)]

class Sigmoid(Module):

    def __init__(self):

        super().__init__()

        self.output = None

    def forward(self, x):

        self.output = []

        for v in x:

            v = max(-500, min(500, v))

            self.output.append(1.0 / (1.0 + math.exp(-v)))

        return self.output

    def backward(self, grad):

        return [g * o * (1 - o) for g, o in zip(grad, self.output)]

class Tanh(Module):

    def __init__(self):

        super().__init__()

        self.output = None

    def forward(self, x):

        self.output = [math.tanh(v) for v in x]

        return self.output

    def backward(self, grad):

        return [g * (1 - o * o) for g, o in zip(grad, self.output)]

In [ ]:
```

### Step 4: Dropout Module

Randomly zeroes elements during training. Scales remaining elements by 1/(1-p) so expected values stay the same. Does nothing during eval.

In [ ]:
```python

class Dropout(Module):

    def __init__(self, p=0.5):

        super().__init__()

        self.p = p

        self.mask = None

    def forward(self, x):

        if not self.training:

            return x

        self.mask = [0.0 if random.random() < self.p else 1.0 / (1 - self.p) for _ in x]

        return [v * m for v, m in zip(x, self.mask)]

    def backward(self, grad):

        if self.mask is None:

            return grad

        return [g * m for g, m in zip(grad, self.mask)]

In [ ]:
```

### Step 5: BatchNorm Module

Normalizes activations to zero mean and unit variance per feature across the batch. Maintains running statistics for eval mode.

In [ ]:
```python

class BatchNorm(Module):

    def __init__(self, size, momentum=0.1, eps=1e-5):

        super().__init__()

        self.size = size

        self.gamma = [1.0] * size

        self.beta = [0.0] * size

        self.gamma_grads = [0.0] * size

        self.beta_grads = [0.0] * size

        self.running_mean = [0.0] * size

        self.running_var = [1.0] * size

        self.momentum = momentum

        self.eps = eps

        self.x_norm = None

        self.std_inv = None

        self.batch_input = None

    def forward_batch(self, batch):

        batch_size = len(batch)

        output_batch = []

        if self.training:

            mean = [0.0] * self.size

            for sample in batch:

                for j in range(self.size):

                    mean[j] += sample[j]

            mean = [m / batch_size for m in mean]

            var = [0.0] * self.size

            for sample in batch:

                for j in range(self.size):

                    var[j] += (sample[j] - mean[j]) ** 2

            var = [v / batch_size for v in var]

            self.std_inv = [1.0 / math.sqrt(v + self.eps) for v in var]

            self.x_norm = []

            self.batch_input = batch

            for sample in batch:

                normed = [(sample[j] - mean[j]) * self.std_inv[j] for j in range(self.size)]

                self.x_norm.append(normed)

                output = [self.gamma[j] * normed[j] + self.beta[j] for j in range(self.size)]

                output_batch.append(output)

            for j in range(self.size):

                self.running_mean[j] = (1 - self.momentum) * self.running_mean[j] + self.momentum * mean[j]

                self.running_var[j] = (1 - self.momentum) * self.running_var[j] + self.momentum * var[j]

        else:

            std_inv = [1.0 / math.sqrt(v + self.eps) for v in self.running_var]

            for sample in batch:

                normed = [(sample[j] - self.running_mean[j]) * std_inv[j] for j in range(self.size)]

                output = [self.gamma[j] * normed[j] + self.beta[j] for j in range(self.size)]

                output_batch.append(output)

        return output_batch

    def forward(self, x):

        result = self.forward_batch([x])

        return result[0]

    def backward(self, grad):

        if self.x_norm is None:

            return grad

        for j in range(self.size):

            self.gamma_grads[j] += self.x_norm[0][j] * grad[j]

            self.beta_grads[j] += grad[j]

        return [grad[j] * self.gamma[j] * self.std_inv[j] for j in range(self.size)]

    def parameters(self):

        params = []

        for j in range(self.size):

            params.append((self.gamma, j, None, self.gamma_grads))

            params.append((self.beta, j, None, self.beta_grads))

        return params

In [ ]:
```

### Step 6: Sequential Container

Chains modules. Forward goes left-to-right, backward goes right-to-left.

In [ ]:
```python

class Sequential(Module):

    def __init__(self, *modules):

        super().__init__()

        self.modules = list(modules)

    def forward(self, x):

        for module in self.modules:

            x = module.forward(x)

        return x

    def backward(self, grad):

        for module in reversed(self.modules):

            grad = module.backward(grad)

        return grad

    def parameters(self):

        params = []

        for module in self.modules:

            params.extend(module.parameters())

        return params

    def train(self):

        self.training = True

        for module in self.modules:

            module.train()

    def eval(self):

        self.training = False

        for module in self.modules:

            module.eval()

In [ ]:
```

### Step 7: Loss Functions

MSE and Binary Cross-Entropy. Each returns the loss value and provides a backward() that returns the gradient.

In [ ]:
```python

class MSELoss:

    def __call__(self, predicted, target):

        self.predicted = predicted

        self.target = target

        n = len(predicted)

        self.loss = sum((p - t) ** 2 for p, t in zip(predicted, target)) / n

        return self.loss

    def backward(self):

        n = len(self.predicted)

        return [2 * (p - t) / n for p, t in zip(self.predicted, self.target)]

class BCELoss:

    def __call__(self, predicted, target):

        self.predicted = predicted

        self.target = target

        eps = 1e-7

        n = len(predicted)

        self.loss = 0

        for p, t in zip(predicted, target):

            p = max(eps, min(1 - eps, p))

            self.loss += -(t * math.log(p) + (1 - t) * math.log(1 - p))

        self.loss /= n

        return self.loss

    def backward(self):

        eps = 1e-7

        n = len(self.predicted)

        grads = []

        for p, t in zip(self.predicted, self.target):

            p = max(eps, min(1 - eps, p))

            grads.append((-t / p + (1 - t) / (1 - p)) / n)

        return grads

In [ ]:
```

### Step 8: SGD and Adam Optimizers

Both take a parameter list and update weights using gradients.

In [ ]:
```python

class SGD:

    def __init__(self, parameters, lr=0.01):

        self.params = parameters

        self.lr = lr

    def step(self):

        for container, i, j, grad_container in self.params:

            if j is not None:

                container[i][j] -= self.lr * grad_container[i][j]

            else:

                container[i] -= self.lr * grad_container[i]

    def zero_grad(self):

        for container, i, j, grad_container in self.params:

            if j is not None:

                grad_container[i][j] = 0.0

            else:

                grad_container[i] = 0.0

class Adam:

    def __init__(self, parameters, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):

        self.params = parameters

        self.lr = lr

        self.beta1 = beta1

        self.beta2 = beta2

        self.eps = eps

        self.t = 0

        self.m = [0.0] * len(parameters)

        self.v = [0.0] * len(parameters)

    def step(self):

        self.t += 1

        for idx, (container, i, j, grad_container) in enumerate(self.params):

            if j is not None:

                g = grad_container[i][j]

            else:

                g = grad_container[i]

            self.m[idx] = self.beta1 * self.m[idx] + (1 - self.beta1) * g

            self.v[idx] = self.beta2 * self.v[idx] + (1 - self.beta2) * g * g

            m_hat = self.m[idx] / (1 - self.beta1 ** self.t)

            v_hat = self.v[idx] / (1 - self.beta2 ** self.t)

            update = self.lr * m_hat / (math.sqrt(v_hat) + self.eps)

            if j is not None:

                container[i][j] -= update

            else:

                container[i] -= update

    def zero_grad(self):

        for container, i, j, grad_container in self.params:

            if j is not None:

                grad_container[i][j] = 0.0

            else:

                grad_container[i] = 0.0

In [ ]:
```

### Step 9: DataLoader

Splits data into batches, optionally shuffles each epoch.

In [ ]:
```python

class DataLoader:

    def __init__(self, data, batch_size=32, shuffle=True):

        self.data = data

        self.batch_size = batch_size

        self.shuffle = shuffle

    def __iter__(self):

        indices = list(range(len(self.data)))

        if self.shuffle:

            random.shuffle(indices)

        for start in range(0, len(indices), self.batch_size):

            batch_indices = indices[start:start + self.batch_size]

            batch = [self.data[i] for i in batch_indices]

            inputs = [item[0] for item in batch]

            targets = [item[1] for item in batch]

            yield inputs, targets

    def __len__(self):

        return (len(self.data) + self.batch_size - 1) // self.batch_size

In [ ]:
```

### Step 10: Train a 4-Layer Network on Circle Classification

Wire everything together. Define a model, pick a loss, pick an optimizer, run the training loop.

In [ ]:
```python

def make_circle_data(n=500, seed=42):

    random.seed(seed)

    data = []

    for _ in range(n):

        x = random.uniform(-2, 2)

        y = random.uniform(-2, 2)

        label = 1.0 if x * x + y * y < 1.5 else 0.0

        data.append(([x, y], [label]))

    return data

def train():

    random.seed(42)

    model = Sequential(

        Linear(2, 16),

        ReLU(),

        Linear(16, 16),

        ReLU(),

        Linear(16, 8),

        ReLU(),

        Linear(8, 1),

        Sigmoid(),

    )

    criterion = BCELoss()

    optimizer = Adam(model.parameters(), lr=0.01)

    data = make_circle_data(500)

    split = int(len(data) * 0.8)

    train_data = data[:split]

    test_data = data[split:]

    loader = DataLoader(train_data, batch_size=16, shuffle=True)

    model.train()

    for epoch in range(100):

        total_loss = 0

        total_correct = 0

        total_samples = 0

        for batch_inputs, batch_targets in loader:

            batch_loss = 0

            for x, t in zip(batch_inputs, batch_targets):

                pred = model.forward(x)

                loss = criterion(pred, t)

                batch_loss += loss

                optimizer.zero_grad()

                grad = criterion.backward()

                model.backward(grad)

                optimizer.step()

                predicted_class = 1.0 if pred[0] >= 0.5 else 0.0

                if predicted_class == t[0]:

                    total_correct += 1

                total_samples += 1

            total_loss += batch_loss

        avg_loss = total_loss / total_samples

        accuracy = total_correct / total_samples * 100

        if epoch % 10 == 0 or epoch == 99:

            print(f"Epoch {epoch:3d} | Loss: {avg_loss:.6f} | Train Accuracy: {accuracy:.1f}%")

    model.eval()

    correct = 0

    for x, t in test_data:

        pred = model.forward(x)

        predicted_class = 1.0 if pred[0] >= 0.5 else 0.0

        if predicted_class == t[0]:

            correct += 1

    test_accuracy = correct / len(test_data) * 100

    print(f"\nTest Accuracy: {test_accuracy:.1f}% ({correct}/{len(test_data)})")

    return model, test_accuracy

In [ ]:
```

## Exercises

In [ ]:
1. Add a `SoftmaxCrossEntropyLoss` class for multi-class classification. Softmax the predictions, compute cross-entropy loss, and handle the combined backward pass. Test it on a 3-class spiral dataset.

2. Implement learning rate scheduling in the optimizer: add a `set_lr()` method and wire in the cosine schedule from Lesson 09. Train the circle classifier with warmup + cosine and compare to constant LR.

3. Add a `save()` and `load()` method to Sequential that serializes all weights to a JSON file and loads them back. Verify that a loaded model produces the same predictions as the original.

4. Implement weight decay (L2 regularization) in the Adam optimizer. Add a `weight_decay` parameter that shrinks weights toward zero each step. Compare training with decay=0 vs decay=0.01.

5. Replace the per-sample training loop with proper mini-batch gradient accumulation: accumulate gradients across all samples in a batch, then divide by batch size and take one optimizer step. Measure whether this changes convergence speed.